# ANTROUTE: train the CNN+LSTM + RADR STGNN, then score every session

Runs on a **GPU** Colab runtime (Runtime -> Change runtime type -> T4 GPU). Use the session you freed by stopping the
frame extraction in notebook A.

**Before you start**
1. `git push` the latest code from your PC (this notebook pulls it).
2. Wait for Drive for Desktop to finish syncing, so your newest `labels.csv` files are in the cloud.
3. Frames folders on Drive: `MMDA_FRAMES_PILOT` (May 4 labels), `MMDA_FRAMES` (notebook A: May 11, 13 ...),
   `MMDA_FRAMES_B` (notebook B: May 18, 20, 22 ...).

Part 1 trains. Part 2 (run only when extraction is complete) scores every session and writes the risk scores.

## 0. Check the GPU

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU - change the runtime type")

## 1. Setup: code, packages, and a fast local copy of the frames

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
!test -d /content/ANTROUTE && git -C /content/ANTROUTE pull || git clone -b dev https://github.com/Traffic-Forecasting-Thesis-Group/ANTROUTE.git /content/ANTROUTE
!pip install -q scipy
!git -C /content/ANTROUTE log --oneline -1

In [ ]:
# Copy the frames from Drive to the fast local disk (training reads thousands of images; Drive is slow).
# Re-run this cell any time you add labels; rsync only copies what changed.
import os
FRAMES = {"pilot": "MMDA_FRAMES_PILOT", "a": "MMDA_FRAMES", "b": "MMDA_FRAMES_B"}   # local name -> Drive folder
os.makedirs("/content/frames", exist_ok=True)
for local, folder in FRAMES.items():
    src = f"/content/drive/MyDrive/{folder}"
    if os.path.isdir(src):
        !rsync -a "{src}/" "/content/frames/{local}/"
        print("copied", folder, "->", f"/content/frames/{local}")
    else:
        print("MISSING on Drive:", folder)

In [ ]:
# What labels do we have, and how would the 70/15/15 split look?
import sys; sys.path.insert(0, "/content/ANTROUTE/traffic_system")
from pathlib import Path
import collections, pandas as pd
from src.data.training_data import (assign_session_splits, describe_split, labelled_sessions, load_frames_table,
                                    load_label_lookup, session_of)
roots = [Path(f"/content/frames/{n}") for n in FRAMES if Path(f"/content/frames/{n}").exists()]
lookup = load_label_lookup(roots, human_only=True)
frames = load_frames_table(roots)
times = pd.to_datetime(frames.loc[frames["frame_path"].isin(lookup), "timestamp"], format="ISO8601")
per_session = collections.Counter(session_of(t) for t in times)
print("your labels per session:", {f"{d:%b %d} {s}": n for (d, s), n in sorted(per_session.items())})
print("\nsplit that --split auto will use (sessions need >= 100 labels):")
print(describe_split(assign_session_splits(labelled_sessions(roots, lookup, 100))))

## 2. Train
The printed table above must show **train / val / test** sessions. With fewer than 3 labelled sessions there is no test set yet.

Baselines to beat on validation macro-F1: always-Heavy 0.24, "the camera's usual label" 0.48, pretrained image features + linear 0.42.

In [ ]:
RUN = "stgnn_v1"          # name of this run; change it for each experiment
EPOCHS = 12
CKPT_DIR = "/content/drive/MyDrive/MMDA_CHECKPOINTS"
!mkdir -p {CKPT_DIR}
!cd /content/ANTROUTE/traffic_system && python scripts/train_stgnn.py \
    --frames-root /content/frames/pilot /content/frames/a /content/frames/b \
    --human-only --split auto --no-text --class-weights \
    --epochs {EPOCHS} --batch-size 4 --num-workers 2 \
    --out {CKPT_DIR}/{RUN}.pt

In [ ]:
# Per-epoch results (also saved next to the checkpoint)
m = pd.read_csv(f"{CKPT_DIR}/{RUN}.metrics.csv")
display(m.round(3))
best = m.loc[m["val_macro_f1"].idxmax()]
print(f"best epoch {int(best['epoch'])}: val accuracy {best['val_accuracy']:.0%}, macro-F1 {best['val_macro_f1']:.2f}   (camera-prior baseline: macro-F1 0.48)")

### Optional ablations (run one at a time, with a new `RUN` name)
```
--no-time-features        # remove the time-in-session input
--no-node-embedding       # remove the per-intersection bias
```
For a text-on run, copy your regenerated `embeddings.pt` to Drive and replace `--no-text` with
`--embeddings /content/drive/MyDrive/MMDA_TEXT/embeddings.pt`.

## 3. Score every session (run when extraction is complete)
Re-run the copy cell in section 1 first, so the frames extracted since then are on the local disk.
Outputs go to Drive: `risk_nodes.csv` (per window and intersection), `risk_edges.csv` (per road edge) and `risk_summary.json`.

In [ ]:
OUT = "/content/drive/MyDrive/MMDA_RISK_SCORES"
!cd /content/ANTROUTE/traffic_system && python scripts/predict_risk.py \
    --checkpoint {CKPT_DIR}/{RUN}.pt \
    --frames-root /content/frames/pilot /content/frames/a /content/frames/b \
    --out-dir {OUT} --edges --batch-size 8 --num-workers 2

In [ ]:
import json
summary = json.load(open(f"{OUT}/risk_summary.json"))
print("windows scored:", summary["windows"], "| sessions:", summary["sessions"])
display(pd.DataFrame(summary["splits"]).T.round(3))
nodes = pd.read_csv(f"{OUT}/risk_nodes.csv")
display(nodes.head(10))
print("mean risk per intersection:"); display(nodes.groupby("intersection")["risk"].mean().round(2).sort_values())